# SLT Grokking Experiment — Colab Runner

**How to use:**
1. Upload the `slt_grokking/` project folder to your Google Drive (or sync via Drive desktop app).
2. Mount Drive in Section 0 and set `DRIVE_PROJECT_PATH`.
3. Run Section 1 (install) once per Colab session.
4. Run the relevant section for your task:
   - Section 2: Training
   - Section 3: LLC calibration
   - Section 4: LLC estimation
   - Section 5: Figures

Checkpoints and results save to Drive automatically — safe across disconnects.

## Section 0: Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── EDIT THIS PATH ──────────────────────────────────────────────────────────
# Path to the slt_grokking/ folder inside your Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/slt_grokking'
# ─────────────────────────────────────────────────────────────────────────────

import sys, os
sys.path.insert(0, DRIVE_PROJECT_PATH + '/src')
os.chdir(DRIVE_PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')
print(f'src on path: {DRIVE_PROJECT_PATH}/src')

## Section 1: Install dependencies (run once per Colab session)

In [ ]:
# transformer_lens and devinterp — the two non-standard deps
%pip install -q transformer_lens devinterp

# Verify devinterp version and available API
import devinterp
print(f'devinterp version: {devinterp.__version__}')

try:
    from devinterp.slt.llc import llc
    print('API: devinterp v2 (llc function)')
    DEVINTERP_V2 = True
except ImportError:
    from devinterp.slt.sampler import estimate_learning_coeff_with_summary
    print('API: devinterp v1 (estimate_learning_coeff_with_summary)')
    DEVINTERP_V2 = False

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}  |  device: {device}')

## Section 2: Training

Run one (ratio, seed) pair. For the full sweep, run this cell 21 times (7 ratios × 3 seeds).
Use `RESUME = True` to continue after a Colab disconnect.

In [ ]:
# ── CONFIGURE ────────────────────────────────────────────────────────────────
RATIO  = 0.50   # addition fraction: one of [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEED   = 0      # seed: 0, 1, or 2
RESUME = False  # set True to pick up from last checkpoint
# ─────────────────────────────────────────────────────────────────────────────

resume_flag = '--resume' if RESUME else ''
cmd = f'python src/train.py --ratio {RATIO} --seed {SEED} {resume_flag}'
print(f'Running: {cmd}')
!{cmd}

## Section 3: LLC Calibration

Run **once** after at least one training run is complete (final checkpoint of ratio=0.50, seed=0).
Inspect the trace output, then update `configs/llc_calibration.yaml` with the best hyperparams.

Good calibration = chains mix (fluctuate without diverging or flatlining).

In [ ]:
# Uses the converged ratio=0.50 seed=0 checkpoint by default
!python src/llc_estimation.py --ratio 0.50 --seed 0 --calibrate

In [ ]:
# Plot calibration traces to visually inspect chain mixing
import numpy as np
import matplotlib.pyplot as plt

traces_path = 'results/metrics/calibration_traces.npy'
traces = np.load(traces_path)
print(f'Traces shape: {traces.shape}  (num_chains × num_draws)')

fig, ax = plt.subplots(figsize=(10, 4))
for i, chain in enumerate(traces):
    ax.plot(chain, lw=0.8, alpha=0.7, label=f'Chain {i}')
ax.set_xlabel('Draw')
ax.set_ylabel('Loss (SGLD)')
ax.set_title('SGLD chain traces — calibration run')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('results/figures/fig_calibration_traces.pdf', bbox_inches='tight')
plt.show()

print()
print('If chains diverge → decrease epsilon (e.g. 1e-5)')
print('If chains barely move → increase epsilon or num_draws')
print('Good: chains fluctuate in a stable band around the mean loss')

In [ ]:
# After visual inspection, update configs/llc_calibration.yaml
# Example — edit values before running:

import yaml
from pathlib import Path

# ── FILL IN calibrated values ─────────────────────────────────────────────
CALIBRATED_EPSILON   = 1e-4
CALIBRATED_NBETA     = 1.0
CALIBRATED_GAMMA     = 10.0
CALIBRATED_NUM_CHAINS = 8
CALIBRATED_NUM_DRAWS  = 500
CALIBRATED_BURNIN     = 100
# ─────────────────────────────────────────────────────────────────────────

cfg = {
    'calibrated': True,
    'epsilon': CALIBRATED_EPSILON,
    'nbeta': CALIBRATED_NBETA,
    'gamma': CALIBRATED_GAMMA,
    'num_chains': CALIBRATED_NUM_CHAINS,
    'num_draws': CALIBRATED_NUM_DRAWS,
    'num_burnin_steps': CALIBRATED_BURNIN,
    'calibration_checkpoint': 'results/checkpoints/ratio_0.50/seed_0/epoch_06000.pt',
    'calibration_date': '2026-XX-XX',
    'calibration_notes': 'chains mixed well',
}

Path('configs/llc_calibration.yaml').write_text(yaml.dump(cfg, default_flow_style=False))
print('Saved configs/llc_calibration.yaml')
print(yaml.dump(cfg))

## Section 4: LLC Estimation

Run after training AND calibration are done. This processes all 100 checkpoints for one (ratio, seed) pair.
Wall-clock estimate: ~2–4 min per checkpoint × 100 = 3–7 hours per (ratio, seed).

**Ask Tair before starting** — this is the expensive compute step (>2h).

In [ ]:
# ── CONFIGURE ────────────────────────────────────────────────────────────────
RATIO = 0.50
SEED  = 0
# ─────────────────────────────────────────────────────────────────────────────

# Verify calibration config is set
import yaml
cfg = yaml.safe_load(open('configs/llc_calibration.yaml'))
if not cfg.get('calibrated'):
    print('ERROR: llc_calibration.yaml not filled in yet. Run Section 3 first.')
else:
    print(f'Using calibrated hyperparams: epsilon={cfg["epsilon"]} nbeta={cfg["nbeta"]} gamma={cfg["gamma"]}')
    print(f'Estimating LLC for ratio={RATIO} seed={SEED}')
    print('This will take several hours. Checkpoints resume automatically.')
    !python src/llc_estimation.py --ratio {RATIO} --seed {SEED}

## Section 5: Generate Figures

In [ ]:
!python src/plotting.py --metrics_dir results/metrics --figures_dir results/figures

# List produced figures
from pathlib import Path
figs = sorted(Path('results/figures').glob('*.pdf'))
for f in figs:
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

## Section 6: Full sweep runner (advanced)

Runs all 7 ratios × 3 seeds sequentially. Only use if you have a long uninterrupted session (12h+).
Training only — does NOT run LLC estimation (that needs a separate approval per PROJECT_CONTEXT.md).

In [ ]:
RATIOS = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEEDS  = [0, 1, 2]
RESUME = True  # always resume — safe to re-run

for ratio in RATIOS:
    for seed in SEEDS:
        print(f'\n{'='*60}')
        print(f'  Training ratio={ratio}  seed={seed}')
        print(f'{'='*60}')
        resume_flag = '--resume' if RESUME else ''
        !python src/train.py --ratio {ratio} --seed {seed} {resume_flag}

print('\nAll training runs complete.')

## Section 7: Quick diagnostics

Run any time to check what's been computed so far.

In [ ]:
from pathlib import Path
import pandas as pd

print('=== Checkpoints ===')
ckpt_base = Path('results/checkpoints')
for ratio_dir in sorted(ckpt_base.glob('ratio_*')):
    for seed_dir in sorted(ratio_dir.glob('seed_*')):
        ckpts = sorted(seed_dir.glob('epoch_*.pt'))
        if ckpts:
            last = int(ckpts[-1].stem.split('_')[1])
            print(f'  {ratio_dir.name}/{seed_dir.name}: {len(ckpts)} checkpoints, last={last}')

print()
print('=== Metrics CSVs ===')
for f in sorted(Path('results/metrics').glob('ratio_*.csv')):
    if '_llc' not in f.name:
        df = pd.read_csv(f)
        print(f'  {f.name}: {len(df)} rows, max_epoch={df["epoch"].max()}')

print()
print('=== LLC CSVs ===')
for f in sorted(Path('results/metrics').glob('*_llc.csv')):
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df)} rows computed')